# 04 - Databricks App: deploy, grant least privilege, and check it end to end

1. Deploy the AppKit app in `app/` with its bundle.
2. Least privilege. The app's service principal gets Lakebase access only: read the synced gold
   schema and own its `committee` schema for decisions. It has no Unity Catalog grants. Genie runs
   on behalf of the signed-in user (`user_api_scopes: dashboards.genie`), so Unity Catalog applies
   that user's own permissions, including the counterparty mask.
3. Call the deployed app: read the committee pack from Lakebase, record a committee decision,
   read that decision back through Unity Catalog, and ask Genie through the app.

In [1]:
import json, logging, os, subprocess, warnings, yaml
os.environ["GRPC_VERBOSITY"] = "NONE"
warnings.filterwarnings("ignore")
logging.getLogger("databricks.sdk").setLevel(logging.ERROR)
import requests
from databricks.connect import DatabricksSession
from databricks.sdk import WorkspaceClient

cfg = yaml.safe_load(open("../config.yml"))
spark = DatabricksSession.builder.profile(cfg["profile"]).serverless(True).getOrCreate()
w = WorkspaceClient(profile=cfg["profile"])
cat, sch, lb_cat = cfg["catalog"], cfg["schema"], cfg["lakebase_catalog"]

def run(*cmd, cwd=None):
    out = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    lines = [l for l in (out.stdout + out.stderr).strip().splitlines() if "http" not in l]  # keep URLs out of the log
    print(lines[-1] if lines else "")
    out.check_returncode()

## Deploy

In [2]:
run("databricks", "bundle", "deploy", "--profile", cfg["profile"], cwd="../app")
run("databricks", "bundle", "run", "app", "--profile", cfg["profile"], cwd="../app")
app = w.apps.get("stress-demo")
sp = app.service_principal_client_id
print("app status:", app.app_status.state.value, "| compute:", app.compute_status.state.value)

Resources: 0 created, 0 changed, 0 deleted, 1 unchanged


✓ App started successfully


app status: RUNNING | compute: ACTIVE


## Least privilege: what the app's identity can do

In [3]:
print("app user API scopes:", app.effective_user_api_scopes)
print("UC grants for the app on catalog", cat, ":", spark.sql(f"SHOW GRANTS `{sp}` ON CATALOG {cat}").count())

# Lakebase: the synced tables are owned by the sync pipeline, so grant read on them explicitly.
run("databricks", "psql", "--project", cfg["lakebase_project"], "--profile", cfg["profile"], "--",
    "-v", "ON_ERROR_STOP=1", "-c", f'GRANT USAGE ON SCHEMA {sch} TO "{sp}"; GRANT SELECT ON ALL TABLES IN SCHEMA {sch} TO "{sp}";')

app user API scopes: ['dashboards.genie', 'iam.access-control:read', 'iam.current-user:read']


UC grants for the app on catalog stress_demo : 0


Connecting to read-write endpoint...


## Call the deployed app

In [4]:
s = requests.Session()
s.headers["Authorization"] = w.config.authenticate()["Authorization"]
base = app.url

pack = s.get(f"{base}/api/stress").json()
print("committee pack from Lakebase:", len(pack["desks"]), "desk rows,", len(pack["scenarios"]), "scenarios")
for d in pack["desks"]:
    if d["limit_breach"]:
        print(f"  breach: {d['scenario_name']} / {d['desk']}: P&L JPY {d['pnl_jpy'] / 1e9:.1f}bn vs limit JPY {d['limit_jpy'] / 1e9:.1f}bn")

r = s.post(f"{base}/api/approvals", json={"scenario_id": "S1", "decision": "SIGNED_OFF",
                                          "note": "All desks within limits; no action needed."})
print("record decision:", r.status_code, {k: r.json()[k] for k in ["scenario_id", "decision", "note"]})

committee pack from Lakebase: 16 desk rows, 4 scenarios
  breach: Yen surges 15% / FX: P&L JPY -11.6bn vs limit JPY 8.0bn
  breach: Credit spreads widen / Credit: P&L JPY -4.9bn vs limit JPY 4.0bn


record decision: 201 {'scenario_id': 'S1', 'decision': 'SIGNED_OFF', 'note': 'All desks within limits; no action needed.'}


## The decision written by the app is visible in Unity Catalog (Lakebase catalog)

In [5]:
spark.sql(f"""
  SELECT scenario_id, decision, note, decided_at FROM {lb_cat}.committee.approvals ORDER BY decided_at
""").show(truncate=False)

+-----------+----------------+------------------------------------------------------------+--------------------------+
|scenario_id|decision        |note                                                        |decided_at                |
+-----------+----------------+------------------------------------------------------------+--------------------------+
|S2         |ACTION_REQUESTED|FX desk to cut USDJPY exposure below limit by next committee|2026-11-16 10:49:06.999307|
|S1         |SIGNED_OFF      |All desks within limits; no action needed.                  |2026-11-16 11:00:07.636717|
|S1         |SIGNED_OFF      |All desks within limits; no action needed.                  |2026-11-16 11:12:23.470244|
|S1         |SIGNED_OFF      |All desks within limits; no action needed.                  |2026-11-16 11:14:37.844861|
+-----------+----------------+------------------------------------------------------------+--------------------------+



## Ask Genie through the app (server-sent events from `/api/genie/default/messages`)

In [6]:
events = []
with s.post(f"{base}/api/genie/default/messages", stream=True,
            json={"content": "Which desks breach their stress loss limit, and by how much?"}) as resp:
    for line in resp.iter_lines(decode_unicode=True):
        if line and line.startswith("data:"):
            events.append(json.loads(line[5:]))
print("event types:", [e.get("type") for e in events])
for e in events:
    if e.get("type") == "message_result":
        msg = e.get("message", e)
        for a in msg.get("attachments", []):
            if a.get("query"):
                print("SQL:", a["query"].get("query", "").strip())
            if a.get("text"):
                print("ANSWER:", a["text"].get("content", "").strip())
    if e.get("type") == "query_result":
        print("ROWS:", e["data"]["result"].get("data_array"))

event types: ['message_start', 'status', 'status', 'status', 'status', 'status', 'message_result', 'query_result']
SQL: SELECT `scenario_name`, `desk`, ROUND(try_divide(-`pnl_jpy`,1e9), 1) AS `loss_jpy_bn`, ROUND(try_divide(`limit_jpy`,1e9), 1) AS `limit_jpy_bn`, ROUND(try_divide(((-`pnl_jpy`) - `limit_jpy`),1e9), 1) AS `breach_amount_jpy_bn`, `limit_utilization` FROM `stress_demo`.`risk`.`gold_desk_stress` WHERE `limit_breach` = TRUE ORDER BY `limit_utilization` DESC
ANSWER: **2 desks** breach their stress loss limit. **FX** breaches by **JPY 3.6bn** in **Yen surges 15%** (loss **JPY 11.6bn** vs limit **JPY 8.0bn**), and **Credit** breaches by **JPY 0.9bn** in **Credit spreads widen** (loss **JPY 4.9bn** vs limit **JPY 4.0bn**). **FX** has the larger breach and higher limit utilization (**1.455** vs **1.225**).
ROWS: [['Yen surges 15%', 'FX', '11.6', '8.0', '3.6', '1.455'], ['Credit spreads widen', 'Credit', '4.9', '4.0', '0.9', '1.225']]
